# Credit Card Transaction and Fraud Pattern Analysis
### End-to-End Banking Risk Analytics, SQL Window Functions, and Rules Engine
**Domain**: Banking and Financial Services  
**Objective**: Detect sophisticated fraud patterns (velocity bursts, carding probe-and-drain, midnight anomalies, and credit limit outliers) with full precision, recall, and financial exposure attribution.

In [1]:
import os
import csv
import json
import sqlite3
import datetime

# Paths to processed warehouse and metrics
base_dir = os.path.dirname(os.getcwd())
db_path = os.path.join(base_dir, "data", "processed", "fraud_warehouse.db")
metrics_path = os.path.join(base_dir, "data", "processed", "model_evaluation_metrics.json")

print(f"Database exists: {os.path.exists(db_path)}")
with open(metrics_path, "r") as f:
    metrics = json.load(f)
print("Model Evaluation Summary Loaded successfully.")

## 1. Executive Fraud KPI Scorecard
Let us review the high-level financial risk metrics calculated by our pipeline.

In [2]:
cm = metrics["classification_metrics"]
fi = metrics["financial_impact"]
sm = metrics["summary"]

print("=" * 65)
print(f"Total Processed Transactions:  {sm["total_transactions"]:,}")
print(f"Total Transacted Volume:      ${sm["total_spend_dollars"]:,.2f}")
print(f"Ground Truth Fraud Cases:     {sm["total_fraud_cases"]:,} ({sm["fraud_rate_pct"]}%)  [Imbalanced]")
print(f"Intercepted Fraud Dollars:    ${fi["prevented_fraud_dollars"]:,.2f} ({fi["fraud_prevention_rate_pct"]}%) ")
print("-" * 65)
print(f"Model Detection Precision:     {cm["precision_pct"]}%")
print(f"Model Detection Recall:        {cm["recall_pct"]}%")
print(f"Ensemble F1-Score:             {cm["f1_score_pct"]}%")
print(f"False Positive Rate (FPR):     {100 - cm["specificity_pct"]:.2f}%")
print("=" * 65)

## 2. SQL Window Function Velocity Queries
We query the relational SQLite / Oracle warehouse to examine sliding transaction velocity checks.

In [3]:
conn = sqlite3.connect(db_path)
cur = conn.cursor()

q_velocity = """
WITH intervals AS (
    SELECT 
        transaction_id, card_id, transaction_timestamp, amount, is_fraud,
        LAG(transaction_timestamp, 1) OVER (PARTITION BY card_id ORDER BY transaction_timestamp) AS prev_ts,
        LAG(amount, 1) OVER (PARTITION BY card_id ORDER BY transaction_timestamp) AS prev_amt
    FROM fact_transactions
)
SELECT 
    transaction_id, card_id, transaction_timestamp, amount, prev_amt,
    ROUND((JULIANDAY(transaction_timestamp) - JULIANDAY(prev_ts)) * 1440, 2) AS elapsed_mins,
    is_fraud
FROM intervals
WHERE prev_ts IS NOT NULL AND (JULIANDAY(transaction_timestamp) - JULIANDAY(prev_ts)) * 1440 <= 15.0
LIMIT 10;
"""
cur.execute(q_velocity)
rows = cur.fetchall()
for r in rows:
    print(f"Tx: {r[0]} | Card: {r[1]} | Time: {r[2]} | Amt: ${r[3]:.2f} | PrevAmt: ${r[4]:.2f} | Delta: {r[5]} mins | Fraud: {r[6]}")

## 3. Merchant Category Risk & Dense Ranking
Evaluating fraud exposure across retail verticals using SQL window ranking functions.

In [4]:
q_rank = """
WITH cat_stats AS (
    SELECT 
        m.category,
        COUNT(t.transaction_id) AS total_tx,
        ROUND(SUM(t.amount), 2) AS total_spend,
        SUM(CASE WHEN t.is_fraud = 1 THEN 1 ELSE 0 END) AS fraud_tx,
        ROUND(SUM(CASE WHEN t.is_fraud = 1 THEN t.amount ELSE 0.0 END), 2) AS fraud_loss,
        ROUND(SUM(CASE WHEN t.is_fraud = 1 THEN 1.0 ELSE 0.0 END) / COUNT(t.transaction_id) * 100, 2) AS fraud_rate
    FROM dim_merchants m
    JOIN fact_transactions t ON m.merchant_id = t.merchant_id
    GROUP BY m.category
)
SELECT 
    category, total_tx, fraud_tx, fraud_loss, fraud_rate,
    DENSE_RANK() OVER (ORDER BY fraud_loss DESC) AS rank_loss
FROM cat_stats
ORDER BY fraud_loss DESC;
"""
cur.execute(q_rank)
for r in cur.fetchall():
    print(f"{r[0]:<18} | Total: {r[1]:<5} | Fraud: {r[2]:<3} | Loss: ${r[3]:<10,.2f} | Rate: {r[4]}% | Rank: {r[5]}")
conn.close()

## 4. Multi-Rule Attribution Breakdown
Reviewing performance of each individual rule implemented in the fraud detection engine.

In [5]:
print("=" * 75)
print(f"{"Rule Name":<35} | {"Flagged":<8} | {"TP":<5} | {"FP":<5} | {"Precision":<10} | {"Caught Loss":<12}")
print("=" * 75)
for r in metrics["rules_breakdown"]:
    print(f"{r["rule_name"]:35} | {r["total_triggered"]:8} | {r["true_positives"]:5} | {r["false_positives"]:5} | {r["precision_pct"]:>8}% | ${r["fraud_dollars_caught"]:>10,.2f}")
print("=" * 75)